In [1]:
from langchain_classic.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from groq import Groq
import os , dotenv, pandas 
dotenv.load_dotenv()
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain 
from langchain_classic.chains import create_history_aware_retriever

from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_core.documents import Document

import dotenv
dotenv.load_dotenv()
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import PromptTemplate

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.output_parsers import *

from langchain_classic.retrievers import MultiQueryRetriever, ContextualCompressionRetriever

from langchain.chat_models import init_chat_model

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, LLMChainFilter

embed = HuggingFaceEmbeddings(model='all-MiniLM-L6-v2')

llm =ChatGoogleGenerativeAI(model='gemini-2.5-flash', api_key=os.getenv("GOOGLE_API_KEY"))


# Load FAISS data
file_path = r"C:\Users\Neel\Desktop\Azure_open_ai\Data\VectStore\Faiss_folder_1"  

vector_store = FAISS.load_local(
    folder_path=file_path,
    embeddings=embed,
    allow_dangerous_deserialization=True
)

# Load FAISS data
file_path = r"C:\Users\Neel\Desktop\Azure_open_ai\Data\VectStore\Faiss_folder_1"  

vector_store = FAISS.load_local(
    folder_path=file_path,
    embeddings=embed,
    allow_dangerous_deserialization=True
)

base_ret=vector_store.as_retriever(search_type='mmr', search_kwargs={"mult_lambda":.5,"k":2})
multi_ret = MultiQueryRetriever.from_llm(retriever=base_ret, llm=llm)

c:\Users\Neel\Desktop\Azure_open_ai\myenv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Neel\Desktop\Azure_open_ai\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5862.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
hyde_prompt =ChatPromptTemplate.from_template("""You are creating this for a Hyde ans generation,
                                              Generate a hypotherical ans for the give question:
                                              Question : {question}
                                              Answer:
                                              
                                              
                                              Instruction Strict:
                                              - Don't explain why you are generating the the ans 
                                              - Use crispt ans to the point 
                                              
                                              """)

In [27]:
question="What is a nural network and what is advantage of this over Machine Learning?"

In [28]:
query_decompose_prompt=PromptTemplate.from_template("""You are a sub-query maker for a rag system 
                                                    generate sub-questions out of the given question
                                                    
                                                    Question : {question}

                                                    Sub-question : 
                                                    
                                                    """)

In [29]:
decopose_chain = query_decompose_prompt | llm | StrOutputParser()
hyde_chain = hyde_prompt | llm| StrOutputParser()

In [8]:
from langchain_core.runnables import *

In [30]:
sub_q= decopose_chain.invoke(question)


In [34]:
sub_q_list=sub_q.strip().split("\n")
print(sub_q_list)

['Sub-question :', '', '*   What is a neural network?', '*   How does a neural network fundamentally work?', '*   What are the advantages of neural networks compared to traditional machine learning algorithms?']


In [35]:
all_ans=[]

for i,doc in enumerate(sub_q_list):
  if i !=0:
    all_ans.extend([hyde_chain.invoke(doc)])


In [36]:
print(all_ans)

['Decision: Approved.', 'A neural network is a computational model inspired by the human brain, designed to recognize patterns and make predictions from data. It consists of interconnected layers of nodes (neurons) that process information and learn through experience.', 'Data enters an input layer, flows through interconnected "neurons" in hidden layers. Each connection has an adjustable weight and each neuron a bias. An activation function processes the signal within each neuron. During training, these weights and biases are adjusted via backpropagation to minimize errors, enabling the network to learn patterns and produce a desired output.', '*   Automatic feature learning, reducing manual feature engineering.\n*   Superior performance on complex, high-dimensional, and unstructured data (e.g., images, text, audio).\n*   Better at modeling highly non-linear relationships.\n*   Scalability and improved performance with larger datasets.\n*   Adaptability through transfer learning.']


In [37]:
input = "/n".join(all_ans)

In [38]:
combine_doc_chain = create_stuff_documents_chain(llm=llm, prompt=ChatPromptTemplate.from_messages([
  ('system', """Generate ans for the user question from the given context only
   Context : {context}"""),
   ('human', "{input}")
])
)

In [39]:
rag_chain = create_retrieval_chain(retriever=multi_ret, combine_docs_chain=combine_doc_chain)

'**Question: What is a neural network and what is advantage of this over Machine Learning?**\n\n**Answer:**\n\nA neural network is a computational model inspired by the human brain, composed of interconnected nodes (neurons) organized in layers that learn and process complex patterns from data.\n\n**Advantages over traditional Machine Learning:**\n\n1.  **Automated Feature Learning:** Automatically extracts and learns relevant features directly from raw data, reducing manual feature engineering.\n2.  **Superior Performance on Complex Data:** Excels in tasks involving unstructured data (images, text, audio) and highly non-linear relationships.\n3.  **Scalability with Data:** Performance generally improves significantly with larger datasets.\n4.  **Hierarchical Pattern Recognition:** Capable of learning multi-level, abstract representations of data.'

In [40]:
response=rag_chain.invoke({'input':input})
response

{'input': 'Decision: Approved./nA neural network is a computational model inspired by the human brain, designed to recognize patterns and make predictions from data. It consists of interconnected layers of nodes (neurons) that process information and learn through experience./nData enters an input layer, flows through interconnected "neurons" in hidden layers. Each connection has an adjustable weight and each neuron a bias. An activation function processes the signal within each neuron. During training, these weights and biases are adjusted via backpropagation to minimize errors, enabling the network to learn patterns and produce a desired output./n*   Automatic feature learning, reducing manual feature engineering.\n*   Superior performance on complex, high-dimensional, and unstructured data (e.g., images, text, audio).\n*   Better at modeling highly non-linear relationships.\n*   Scalability and improved performance with larger datasets.\n*   Adaptability through transfer learning.',

In [41]:
print(response['input'])

Decision: Approved./nA neural network is a computational model inspired by the human brain, designed to recognize patterns and make predictions from data. It consists of interconnected layers of nodes (neurons) that process information and learn through experience./nData enters an input layer, flows through interconnected "neurons" in hidden layers. Each connection has an adjustable weight and each neuron a bias. An activation function processes the signal within each neuron. During training, these weights and biases are adjusted via backpropagation to minimize errors, enabling the network to learn patterns and produce a desired output./n*   Automatic feature learning, reducing manual feature engineering.
*   Superior performance on complex, high-dimensional, and unstructured data (e.g., images, text, audio).
*   Better at modeling highly non-linear relationships.
*   Scalability and improved performance with larger datasets.
*   Adaptability through transfer learning.


In [42]:
print(response['answer'])

Based on the provided context, a good explanation of neural networks would contain:

*   An explanation of the algorithm, in plain English.
